In [685]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize._minimize as minimize 
import numpy.linalg as linal
import scipy.io
from scipy.stats.distributions import chi2

In [686]:
data = scipy.io.loadmat('arx.mat')

In [687]:
u   = data['umeas']
y   = data['ymeas']

std_u = data['stdeu'][0,0]
std_y = data['stdey'][0,0]

mean_shift = lambda x : x - x.mean(axis=0)

In [688]:
Z = np.zeros((1,12))

for i in range(5,1024) : 
    indices = [ i-j for j in range(0,6) ] 
    row = np.append( y[indices], u[indices] )[np.newaxis]
    Z = np.append(Z, row,axis=0)

Z = np.delete(Z,0,axis=0)

In [689]:
Zms = mean_shift(Z)

In [690]:
indices

[1023, 1022, 1021, 1020, 1019, 1018]

In [691]:
U,S,V = linal.svd(Zms/np.sqrt(1019), full_matrices=False)
V = V.T

In [692]:
S**2

array([7.16296767, 6.73717113, 6.1776149 , 4.87994912, 4.63652445,
       4.43460246, 1.16441918, 1.09428521, 0.42061272, 0.30525355,
       0.30018608, 0.1877151 ])

In [693]:
coeffs = np.round(V[:, 11]/V[0, 11], 3)

In [694]:
coeffs

array([ 1.   , -0.03 , -0.042, -0.597, -0.024, -0.014,  0.025,  0.065,
       -2.049,  0.068,  0.014,  2.016])

----

In [917]:
L = 6
Zl = np.zeros((1,2*(L+1)))

for i in range(L,1024) : 
    indices = [ i-j for j in range(0,L+1) ] 
    row = np.append( y[indices], u[indices] )[np.newaxis]
    Zl = np.append(Zl, row,axis=0)

Zl = np.delete(Zl,0,axis=0)

In [918]:
Ul, Sl, Vl = linal.svd(Zl/np.sqrt(1024 - L), full_matrices=False)
eigenvalues = Sl**2

p = 2*(L+1)
d = p-1
N = 1024 - L 

print("STATISTICAL TESTING :")
while True : 
    n_dash = N - (2*p+11)/6
    l_dash = eigenvalues[ p-d : ].sum()/d 

    tau    = n_dash * ( d * np.log(l_dash) - np.log(eigenvalues[ p-d : ]).sum())

    fdom   = 0.5*(d+2)*(d-1)
    chi    = chi2.ppf(0.95, df=fdom)

    # print("-----------------------------------------------------")
    # print(f"Chi-square at 95% and {int(fdom)} dof \t\t: {np.round(chi,4)}")
    # print(f"The computed statistic value is \t: {np.round(tau,4)}")

    if (tau <= chi) : 
        break 
    else : 
        d = d - 1 

print("-----------------------------------------------------",end='\n\n')
print (f"Number of constraints after performing the above test is : {d}")
print (f"Degree of the System is : {L-d+1}")


STATISTICAL TESTING :
-----------------------------------------------------

Number of constraints after performing the above test is : 2
Degree of the System is : 5


* All the values for L, returned 5 as the degree of the system after performing hypothesis testing 
---

In [919]:
C = np.diag(np.append(std_y*np.ones(L+1), std_u*np.ones(L+1)))

In [920]:
Us, Ss, Vs = linal.svd(Zl@linal.inv(C)/np.sqrt(N), full_matrices=False)
Vs = Vs.T

In [921]:
eigenvalues = Ss**2
d = p - 1 

print("STATISTICAL TESTING :")
while True : 
    n_dash = N - (2*p+11)/6
    l_dash = eigenvalues[ p-d : ].sum()/d 

    tau    = n_dash * ( d * np.log(l_dash) - np.log(eigenvalues[ p-d : ]).sum())

    fdom   = 0.5*(d+2)*(d-1)
    chi    = chi2.ppf(0.95, df=fdom)

    # print("-----------------------------------------------------")
    # print(f"Chi-square at 95% and {int(fdom)} dof \t\t: {np.round(chi,4)}")
    # print(f"The computed statistic value is \t: {np.round(tau,4)}")

    if (tau <= chi) : 
        break 
    else : 
        d = d - 1 

print("-----------------------------------------------------",end='\n\n')
print (f"Number of constraints after performing the above test is : {d}")
print (f"Degree of the System is : {L-d+1}")

STATISTICAL TESTING :
-----------------------------------------------------

Number of constraints after performing the above test is : 2
Degree of the System is : 5


In [922]:
eigenvalues[-d:]

array([1.07511885, 0.99248333])

* By scaling the data matrix using the error standard deviation we can estimate the correct degree of the system for larger values of L 
* When unscaled, the data matrix cannot find the correct degree for L values outside 10 to 15 

---

In [923]:
L = 5
Zn = np.zeros((1,2*(L+1)))

for i in range(L,1024) : 
    indices = [ i-j for j in range(0,L+1) ] 
    row = np.append( y[indices], u[indices] )[np.newaxis]
    Zn = np.append(Zn, row,axis=0)

Zn = np.delete(Zn,0,axis=0)

Cn = np.diag(np.append(std_y*np.ones(L+1), std_u*np.ones(L+1)))

p = 2*(L+1)
d = p-1
N = 1024 - L 

Un, Sn, Vn = linal.svd(mean_shift(Zn)@linal.inv(Cn)/np.sqrt(N), full_matrices=False)
coeffs = Vn[-1,:]

In [924]:
coeffs = coeffs@linal.inv(Cn)
coeffs/coeffs[0]

## To compute the coefficients from the last eigenvector, scale back to the unscaled domain using A@inv(C)
## Where A is the last principal component and C is the cholesky decomposition of the error covariance matrix

array([ 1.        , -0.02664311, -0.05368237, -0.57371373, -0.02278538,
       -0.01748476,  0.0244667 ,  0.05642807, -1.96607459,  0.05324383,
        0.04334204,  1.8871865 ])

In [925]:
import random

In [926]:
params = np.zeros((1,12))

for i in range(100) : 

    rand_list = []
    for i in range(700) : 
        rand_list.append(random.randrange(0,1018))

    Zt = Zn[rand_list,:]

    L = 5
    N = 700

    Cn = np.diag(np.append(std_y*np.ones(L+1), std_u*np.ones(L+1)))

    Un, Sn, Vn = linal.svd(mean_shift(Zt)@linal.inv(Cn)/np.sqrt(N), full_matrices=False)
    coeffst = Vn[-1,:]

    # Calculating the coefficients
    coeffst = coeffst@linal.inv(Cn)
    row    = coeffst/coeffst[0]
    params = np.append(params, row [np.newaxis], axis=0)

params = np.delete(params, 0, axis=0)

In [927]:
mean = params.mean(axis=0)

In [928]:
ll = params.mean(axis=0) - 2*params.std(axis=0)
ul = params.mean(axis=0) + 2*params.std(axis=0)

In [929]:
np.round(np.append(ll [np.newaxis],ul [np.newaxis],axis=0),3)

array([[ 1.   , -0.231, -0.266, -0.725, -0.087, -0.081, -0.074, -0.051,
        -2.066, -0.505, -0.443,  1.572],
       [ 1.   ,  0.229,  0.18 , -0.424,  0.055,  0.046,  0.141,  0.157,
        -1.852,  0.508,  0.485,  2.205]])

In [930]:
for i in range(12) : 
    if 0 >= ll[i] and 0 <= ul[i] :
        if i >= 6 : 
            print(f"{i-6} : True") 
        else :
            print(f"{i} : True")
    else :
        if i >= 6 : 
            print(f"{i-6} : False") 
        else :
            print(f"{i} : False")

0 : False
1 : True
2 : True
3 : False
4 : True
5 : True
0 : True
1 : True
2 : False
3 : True
4 : True
5 : False


---


In [941]:
Us, Ss, Vs = linal.svd(Zl@linal.inv(C)/np.sqrt(1024-6), full_matrices=False)
Ss**2

array([22.89180267, 21.52827376, 18.80769059, 17.0195671 , 15.76814941,
       12.16954701, 11.7877952 ,  8.21811795,  7.71199639,  2.24881164,
        1.88739867,  1.62296858,  1.07511885,  0.99248333])

In [951]:
As = np.round(Vs[-2:,:]@linal.inv(C),3)

In [952]:
As

array([[-5.460e-01, -5.780e-01,  4.800e-02,  3.310e-01,  3.280e-01,
         2.500e-02, -1.500e-02,  2.000e-02, -4.700e-02,  1.050e+00,
         1.142e+00, -6.000e-02, -1.033e+00, -1.059e+00],
       [-6.100e-01,  5.400e-01,  2.400e-02,  2.520e-01, -3.100e-01,
         1.000e-03, -4.900e-02, -4.500e-02, -1.800e-02,  1.225e+00,
        -1.056e+00, -1.200e-02, -9.830e-01,  1.046e+00]])

In [955]:
As[0]/As[0,1]

array([ 0.94463668,  1.        , -0.08304498, -0.57266436, -0.56747405,
       -0.0432526 ,  0.02595156, -0.03460208,  0.08131488, -1.816609  ,
       -1.97577855,  0.10380623,  1.78719723,  1.83217993])

In [956]:
As[1]/As[1,0]

array([ 1.00000000e+00, -8.85245902e-01, -3.93442623e-02, -4.13114754e-01,
        5.08196721e-01, -1.63934426e-03,  8.03278689e-02,  7.37704918e-02,
        2.95081967e-02, -2.00819672e+00,  1.73114754e+00,  1.96721311e-02,
        1.61147541e+00, -1.71475410e+00])